<a href="https://colab.research.google.com/github/Orliluq/document_intelligence_docling/blob/main/Document_Intelligence_Docling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tu PDF no es texto: cómo preparar documentos complejos para la IA

**Document Intelligence con Python, Docling y Google Colab**

Este notebook acompaña el artículo técnico. Construiremos un flujo reproducible que convierte un PDF en información estructurada: texto en orden de lectura, tablas en `pandas`, archivos Markdown y JSON, y recortes de imágenes/figuras cuando el documento las contiene.

> **No necesitas subir un PDF para probarlo.** Primero generaremos uno de ejemplo con texto, tabla y gráfico. Al final podrás cambiar una variable para cargar tu propio archivo.

## Qué aprenderás

1. Crear un PDF de prueba sin depender de archivos externos.
2. Configurar Docling para procesar un PDF digital.
3. Convertir el documento a Markdown y JSON.
4. Extraer tablas como DataFrames y exportarlas a CSV.
5. Guardar imágenes detectadas y validar los resultados.

**Entorno:** Google Colab, Python 3.10 o superior. El notebook fija Docling en `2.151.1` para que el ejemplo sea reproducible. Si el proyecto publica una nueva versión, revisa las notas de cambios antes de actualizarla.

## 1. Instalar las dependencias

La instalación incluye Docling, además de las librerías que utilizaremos para crear el PDF de ejemplo y analizar las tablas.

La primera ejecución de Docling puede tardar porque necesita descargar y preparar modelos. Después, esos recursos pueden quedar almacenados en caché en el entorno.

In [2]:
pip -q install "docling==2.151.1" reportlab matplotlib pandas

## 2. Importar las librerías

Usaremos:

- `DocumentConverter`: punto de entrada para convertir documentos.
- `PdfPipelineOptions`: opciones específicas del procesamiento de PDF.
- `InputFormat` y `PdfFormatOption`: para configurar el flujo de PDF de forma explícita.
- `pandas`: para trabajar con tablas.
- `reportlab` y `matplotlib`: para construir nuestro PDF de prueba.

In [3]:
from pathlib import Path
import json
import re
import shutil
import time

import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image as ReportLabImage, PageBreak
)

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

from importlib.metadata import version
print("Docling:", version("docling"))
print("Pandas:", pd.__version__)

Docling: 2.151.1
Pandas: 2.2.3


## 3. Crear un PDF de ejemplo

Para probar extracción y estructura necesitamos un documento que contenga más de un párrafo. Vamos a generar un pequeño informe ficticio con:

- título y subtítulos;
- párrafos en español;
- una tabla con indicadores mensuales;
- un gráfico insertado como imagen.

Los datos son inventados y solo sirven para la demostración.

In [4]:
BASE_DIR = Path("/content/document_intelligence_demo")
BASE_DIR.mkdir(parents=True, exist_ok=True)

PDF_DEMO = BASE_DIR / "informe_comunidad_demo.pdf"
CHART_PATH = BASE_DIR / "grafico_sentimiento.png"

# 1) Crear un gráfico PNG que luego insertaremos dentro del PDF.
meses = ["Enero", "Febrero", "Marzo", "Abril"]
positivos = [72, 90, 83, 110]
neutrales = [30, 35, 32, 38]
negativos = [18, 25, 23, 23]

plt.figure(figsize=(8, 3.8))
plt.plot(meses, positivos, marker="o", label="Positivos")
plt.plot(meses, neutrales, marker="o", label="Neutrales")
plt.plot(meses, negativos, marker="o", label="Negativos")
plt.title("Comentarios por sentimiento")
plt.xlabel("Mes")
plt.ylabel("Cantidad de comentarios")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(CHART_PATH, dpi=160, bbox_inches="tight")
plt.close()

# 2) Construir un PDF con texto, tabla y el gráfico.
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    name="DemoTitle", parent=styles["Title"], fontSize=20,
    leading=25, alignment=TA_LEFT, textColor=colors.HexColor("#17324D"),
    spaceAfter=10
))
styles.add(ParagraphStyle(
    name="DemoHeading", parent=styles["Heading2"], fontSize=13,
    leading=17, textColor=colors.HexColor("#1D607A"), spaceBefore=10,
    spaceAfter=6
))
styles.add(ParagraphStyle(
    name="DemoBody", parent=styles["BodyText"], fontSize=9.5,
    leading=14, spaceAfter=7
))

story = [
    Paragraph("Informe de retroalimentación comunitaria", styles["DemoTitle"]),
    Paragraph("Documento de demostración para extracción estructural con Docling", styles["Italic"]),
    Spacer(1, 0.25 * cm),
    Paragraph("1. Contexto", styles["DemoHeading"]),
    Paragraph(
        "Este informe reúne comentarios ficticios de una comunidad tecnológica. "
        "El objetivo del ejercicio es convertir un PDF en datos que puedan "
        "consultarse, analizarse y reutilizarse en una aplicación.",
        styles["DemoBody"]
    ),
    Paragraph(
        "Un PDF puede contener texto seleccionable, elementos gráficos y tablas. "
        "La posición de cada elemento aporta información sobre su función y "
        "sobre el orden en que debería leerse.",
        styles["DemoBody"]
    ),
    Paragraph("2. Indicadores mensuales", styles["DemoHeading"]),
]

table_data = [
    ["Mes", "Comentarios", "Positivos", "Neutrales", "Negativos"],
    ["Enero", "120", "72", "30", "18"],
    ["Febrero", "150", "90", "35", "25"],
    ["Marzo", "138", "83", "32", "23"],
    ["Abril", "171", "110", "38", "23"],
]
report_table = Table(table_data, repeatRows=1, colWidths=[2.6*cm, 2.7*cm, 2.5*cm, 2.5*cm, 2.5*cm])
report_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#17324D")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, -1), 8.5),
    ("ALIGN", (1, 1), (-1, -1), "CENTER"),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#B8C6D1")),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F1F5F8")]),
    ("TOPPADDING", (0, 0), (-1, -1), 7),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
]))
story.extend([
    report_table,
    Spacer(1, 0.25 * cm),
    Paragraph("3. Visualización", styles["DemoHeading"]),
    Paragraph(
        "El gráfico representa la evolución mensual de los comentarios "
        "positivos, neutrales y negativos. En un flujo documental real, "
        "la detección de una figura no significa que su contenido ya haya "
        "sido interpretado semánticamente.",
        styles["DemoBody"]
    ),
    ReportLabImage(str(CHART_PATH), width=15.5*cm, height=7.4*cm),
    Spacer(1, 0.2 * cm),
    Paragraph("4. Nota metodológica", styles["DemoHeading"]),
    Paragraph(
        "Los valores de esta muestra son ficticios. Antes de usar datos "
        "extraídos en un proceso de negocio, verifica visualmente el PDF "
        "original y contrasta las cifras con una fuente confiable.",
        styles["DemoBody"]
    ),
])

pdf_doc = SimpleDocTemplate(
    str(PDF_DEMO), pagesize=A4,
    rightMargin=1.7*cm, leftMargin=1.7*cm,
    topMargin=1.5*cm, bottomMargin=1.5*cm
)
pdf_doc.build(story)

print(f"PDF de demostración creado: {PDF_DEMO}")
print(f"Tamaño: {PDF_DEMO.stat().st_size:,} bytes")

PDF de demostración creado: /content/document_intelligence_demo/informe_comunidad_demo.pdf
Tamaño: 62,641 bytes


## 4. Elegir el PDF que vamos a procesar

Por defecto utilizaremos el PDF de ejemplo recién creado. Para procesar un documento propio, cambia `USE_MY_PDF` a `True` y vuelve a ejecutar esta celda. Colab abrirá un cuadro para subir el archivo.

**Privacidad:** Colab ejecuta el código en un entorno alojado. No subas documentos confidenciales, personales o sujetos a restricciones si no tienes autorización para procesarlos allí.

In [5]:
USE_MY_PDF = False  # Cambia a True para cargar tu propio PDF.

if USE_MY_PDF:
    from google.colab import files
    uploaded = files.upload()
    pdf_candidates = [name for name in uploaded if name.lower().endswith(".pdf")]
    if not pdf_candidates:
        raise ValueError("No se ha subido ningún archivo PDF.")
    PDF_PATH = Path(pdf_candidates[0])
else:
    PDF_PATH = PDF_DEMO

if not PDF_PATH.exists():
    raise FileNotFoundError(f"No existe el archivo: {PDF_PATH}")

if PDF_PATH.suffix.lower() != ".pdf":
    raise ValueError("Este ejemplo espera un archivo con extensión .pdf")

print("PDF seleccionado:", PDF_PATH)
print("Tamaño:", f"{PDF_PATH.stat().st_size:,} bytes")

PDF seleccionado: /content/document_intelligence_demo/informe_comunidad_demo.pdf
Tamaño: 62,641 bytes


## 5. Configurar Docling

El comportamiento de Docling se puede ajustar según el documento:

- `do_ocr=False`: para el PDF de prueba, que contiene texto digital seleccionable. Evita ejecutar OCR innecesariamente.
- `do_ocr=True`: para PDF escaneados o páginas que sean imágenes. OCR intenta reconocer el texto de esas páginas.
- `do_table_structure=True`: activa la reconstrucción de tablas.
- `generate_picture_images=True`: conserva imágenes de figuras para poder exportarlas.
- `generate_page_images=True`: conserva imágenes de página; es útil para inspección visual, pero aumenta el trabajo y el almacenamiento.

Para un PDF escaneado, cambia `USE_OCR` a `True`. El paquete estándar de Docling incluye RapidOCR; también admite otros motores de OCR configurables.

In [6]:
USE_OCR = False  # Cambia a True si el PDF es escaneado o no tiene capa de texto.

pipeline_options = PdfPipelineOptions(
    do_ocr=USE_OCR,
    do_table_structure=True,
    generate_picture_images=True,
    generate_page_images=True,
)
pipeline_options.images_scale = 1.5

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

print("Configuración lista. OCR activado:", USE_OCR)

Configuración lista. OCR activado: False


## 6. Convertir el PDF

`DocumentConverter` recibe la ruta del archivo y devuelve un resultado de conversión. El documento estructurado queda disponible en `conversion_result.document`.

La primera conversión puede ser más lenta porque Docling prepara los modelos. Reutiliza el mismo `converter` para procesar varios documentos en el mismo entorno.

In [7]:
start = time.perf_counter()
conversion_result = converter.convert(str(PDF_PATH))
elapsed = time.perf_counter() - start

document = conversion_result.document

print("Estado de conversión:", conversion_result.status)
print("Páginas detectadas:", len(document.pages))
print("Tablas detectadas:", len(document.tables))
print(f"Tiempo de conversión: {elapsed:.1f} segundos")

if getattr(conversion_result, "errors", None):
    print("Avisos o errores reportados durante la conversión:")
    for conversion_error in conversion_result.errors:
        print("-", conversion_error)

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Estado de conversión: ConversionStatus.SUCCESS
Páginas detectadas: 1
Tablas detectadas: 1
Tiempo de conversión: 45.5 segundos


## 7. Inspeccionar el contenido en orden de lectura

Un `DoclingDocument` no es solamente una cadena larga. Conserva elementos y su estructura. Podemos recorrerlos con `iterate_items()` y observar los textos que Docling ha detectado.

Esto permite comprobar si los títulos, párrafos y otros elementos aparecen en un orden razonable antes de enviar el contenido a otro sistema.

In [8]:
from itertools import islice

for position, (item, level) in enumerate(islice(document.iterate_items(), 20), start=1):
    label = getattr(item, "label", None)
    label = getattr(label, "value", str(label)) if label is not None else type(item).__name__
    text_value = getattr(item, "text", "")
    preview = re.sub(r"\s+", " ", str(text_value)).strip()
    print(f"{position:02d}. nivel={level} | tipo={label} | {preview[:180]}")

01. nivel=1 | tipo=section_header | Informe de retroalimentación comunitaria
02. nivel=1 | tipo=text | Documento de demostración para extracción estructural con Docling
03. nivel=1 | tipo=section_header | 1. Contexto
04. nivel=1 | tipo=text | Este informe reúne comentarios ficticios de una comunidad tecnológica. El objetivo del ejercicio es convertir un PDF en datos que puedan consultarse, analizarse y reutilizarse en u
05. nivel=1 | tipo=text | Un PDF puede contener texto seleccionable, elementos gráficos y tablas. La posición de cada elemento aporta información sobre su función y sobre el orden en que debería leerse.
06. nivel=1 | tipo=section_header | 2. Indicadores mensuales
07. nivel=1 | tipo=table | 
08. nivel=1 | tipo=section_header | 3. Visualización
09. nivel=1 | tipo=text | El gráfico representa la evolución mensual de los comentarios positivos, neutrales y negativos. En un flujo documental real, la detección de una figura no significa que su contenid
10. nivel=1 | tipo=pictu

## 8. Exportar a Markdown y JSON

**Markdown** es cómodo para leer, revisar o alimentar un flujo de recuperación de información. **JSON** conserva una representación estructurada más adecuada para que otra aplicación pueda recorrer los elementos y sus metadatos.

No son formatos intercambiables: Markdown es práctico para lectura; JSON es más útil cuando necesitamos procesar campos y objetos de forma programática.

In [9]:
OUTPUT_DIR = Path("/content/resultados_docling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

markdown_text = document.export_to_markdown()
markdown_path = OUTPUT_DIR / "documento_extraido.md"
markdown_path.write_text(markdown_text, encoding="utf-8")

json_path = OUTPUT_DIR / "documento_estructurado.json"
with json_path.open("w", encoding="utf-8") as json_file:
    json.dump(document.export_to_dict(), json_file, ensure_ascii=False, indent=2, default=str)

print("Markdown guardado:", markdown_path)
print("JSON guardado:", json_path)
print("\nVista previa del Markdown:\n")
print(markdown_text[:2500])

Markdown guardado: /content/resultados_docling/documento_extraido.md
JSON guardado: /content/resultados_docling/documento_estructurado.json

Vista previa del Markdown:

## Informe de retroalimentación comunitaria

Documento de demostración para extracción estructural con Docling

## 1. Contexto

Este informe reúne comentarios ficticios de una comunidad tecnológica. El objetivo del ejercicio es convertir un PDF en datos que puedan consultarse, analizarse y reutilizarse en una aplicación.

Un PDF puede contener texto seleccionable, elementos gráficos y tablas. La posición de cada elemento aporta información sobre su función y sobre el orden en que debería leerse.

## 2. Indicadores mensuales

| Mes     |   Comentarios |   Positivos |   Neutrales |   Negativos |
|---------|---------------|-------------|-------------|-------------|
| Enero   |           120 |          72 |          30 |          18 |
| Febrero |           150 |          90 |          35 |          25 |
| Marzo   |         

## 9. Extraer las tablas como DataFrames

Docling permite convertir cada tabla detectada en un `pandas.DataFrame` mediante `table.export_to_dataframe(doc=document)`. A partir de ahí podemos inspeccionarla, transformarla y exportarla como CSV.

El reconocimiento de tablas puede fallar o variar según la calidad del PDF, la resolución, las celdas combinadas y la complejidad del diseño. Por eso guardamos y revisamos el resultado, en lugar de asumir que siempre será perfecto.

In [10]:
extracted_tables = []

for table_number, table in enumerate(document.tables, start=1):
    dataframe = table.export_to_dataframe(doc=document)
    extracted_tables.append(dataframe)

    csv_path = OUTPUT_DIR / f"tabla_{table_number}.csv"
    dataframe.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"\nTabla {table_number} — {dataframe.shape[0]} filas x {dataframe.shape[1]} columnas")
    display(dataframe)
    print("CSV guardado:", csv_path)

if not extracted_tables:
    print("No se detectaron tablas. Revisa el PDF y la configuración de estructura de tablas.")


Tabla 1 — 4 filas x 5 columnas


,Mes,Comentarios,Positivos,Neutrales,Negativos
0,Enero,120,72,30,18
1,Febrero,150,90,35,25
2,Marzo,138,83,32,23
3,Abril,171,110,38,23


CSV guardado: /content/resultados_docling/tabla_1.csv


## 10. Exportar imágenes de figuras y tablas

Docling puede conservar imágenes de página, figuras y tablas. Aquí guardaremos recortes de elementos gráficos encontrados durante el recorrido del documento.

**Importante:** guardar un recorte no significa que el sistema haya entendido su significado. Para interpretar el contenido de un gráfico —por ejemplo, explicar tendencias o comparar series— hace falta una etapa adicional de análisis visual, un modelo multimodal o una extracción específica de sus datos.

In [11]:
from docling_core.types.doc import PictureItem, TableItem

picture_count = 0
table_image_count = 0
page_image_count = 0

# Guardar imágenes de página cuando fueron generadas por el pipeline.
for _page_key, page in document.pages.items():
    page_image = getattr(page, "image", None)
    if page_image is not None and getattr(page_image, "pil_image", None) is not None:
        page_image_count += 1
        page_number = getattr(page, "page_no", page_image_count)
        page_path = OUTPUT_DIR / f"pagina_{page_number}.png"
        page_image.pil_image.save(page_path, format="PNG")
        print("Página guardada:", page_path)

for element, _level in document.iterate_items():
    if isinstance(element, PictureItem):
        picture_count += 1
        image = element.get_image(document)
        if image is not None:
            image_path = OUTPUT_DIR / f"figura_{picture_count}.png"
            image.save(image_path, format="PNG")
            print("Figura guardada:", image_path)

    elif isinstance(element, TableItem):
        table_image_count += 1
        image = element.get_image(document)
        if image is not None:
            image_path = OUTPUT_DIR / f"tabla_visual_{table_image_count}.png"
            image.save(image_path, format="PNG")
            print("Recorte de tabla guardado:", image_path)

print("Páginas exportadas:", page_image_count)
print("Figuras encontradas:", picture_count)
print("Tablas encontradas en el recorrido:", table_image_count)

Página guardada: /content/resultados_docling/pagina_1.png
Recorte de tabla guardado: /content/resultados_docling/tabla_visual_1.png
Figura guardada: /content/resultados_docling/figura_1.png
Páginas exportadas: 1
Figuras encontradas: 1
Tablas encontradas en el recorrido: 1


## 11. Validaciones mínimas

Antes de utilizar el contenido en un buscador semántico, un RAG, un agente o un proceso de negocio, conviene aplicar controles básicos.

En esta demostración comprobaremos que:

1. La conversión produjo contenido Markdown.
2. El JSON se puede volver a abrir.
3. Los CSV generados tienen contenido.
4. El archivo original sigue disponible para comparar manualmente.

Estas validaciones no certifican que cada número o cada fragmento se haya extraído correctamente. Para documentos críticos, añade una revisión humana y reglas de negocio específicas.

In [12]:
# 1) El texto extraído no debe estar vacío.
assert markdown_text.strip(), "La conversión no produjo texto Markdown."

# 2) Comprobar que el JSON se puede leer.
with json_path.open("r", encoding="utf-8") as json_file:
    json_data = json.load(json_file)
assert isinstance(json_data, dict), "El JSON no tiene la estructura esperada."

# 3) Revisar los CSV generados.
csv_files = sorted(OUTPUT_DIR.glob("tabla_*.csv"))
for csv_file in csv_files:
    checked_df = pd.read_csv(csv_file)
    print(csv_file.name, "->", checked_df.shape)

print("\nValidaciones básicas completadas.")
print("Ojo: la exactitud semántica debe comprobarse contra el PDF original.")

tabla_1.csv -> (4, 5)

Validaciones básicas completadas.
Ojo: la exactitud semántica debe comprobarse contra el PDF original.


## 12. Empaquetar los resultados y descargarlos

Agruparemos los archivos generados en un ZIP para descargarlos desde Colab. El paquete puede incluir el Markdown, el JSON, los CSV y las imágenes detectadas.

In [13]:
zip_base = Path("/content/resultados_docling")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)
print("Paquete creado:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Estás fuera de Google Colab. Descarga el ZIP desde el panel de archivos del entorno.")

Paquete creado: /content/resultados_docling.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. ¿Qué cambia cuando el PDF es escaneado?

Un PDF digital suele contener una capa de texto que puede extraerse directamente. Un PDF escaneado puede ser, en la práctica, una colección de imágenes: al seleccionar una frase no aparece texto porque no existe una capa textual.

Para ese caso, cambia:

```python
USE_OCR = True
```

Y vuelve a ejecutar desde la celda de configuración de Docling. El OCR intenta reconocer los caracteres de las páginas escaneadas. La calidad dependerá de la resolución, inclinación, contraste, tipografía, idioma y estado del original.

Si el documento es multilingüe o contiene escrituras específicas, revisa el motor de OCR y su configuración de idiomas en la documentación oficial de Docling.

## 14. Problemas frecuentes

| Síntoma | Qué revisar |
|---|---|
| La conversión tarda mucho la primera vez | Descarga y caché de modelos; prueba con un PDF corto. |
| El PDF escaneado no devuelve texto | Activa OCR con `USE_OCR = True`. |
| No aparecen tablas | Revisa si la tabla es una imagen, la resolución y `do_table_structure=True`. |
| Las columnas aparecen mezcladas | Contrasta la tabla con la página original; prueba otras opciones de estructura de tablas. |
| No aparece ninguna figura | Confirma `generate_picture_images=True` y revisa si el PDF contiene imágenes detectables. |
| El resultado no respeta el orden de lectura | Inspecciona los elementos y la página original; los diseños multicolumna complejos pueden requerir ajustes. |

## 15. Qué hemos construido

El flujo convierte un PDF en varias representaciones reutilizables:

- **Markdown:** lectura y revisión humana.
- **JSON:** estructura para integraciones y procesos posteriores.
- **CSV/DataFrame:** análisis tabular.
- **PNG:** inspección visual de figuras y tablas.

El siguiente paso podría ser añadir chunking, metadatos por página, embeddings y un sistema RAG. Pero la calidad de esas etapas dependerá de la calidad de la extracción inicial.

> Una aplicación de IA no puede recuperar con fiabilidad información que nunca fue extraída correctamente. Antes de pedirle respuestas al modelo, preparemos bien el documento.

---

### Referencias oficiales

- [Docling — Quickstart](https://docling-project.github.io/docling/getting_started/quickstart/)
- [Docling — Installation](https://docling-project.github.io/docling/getting_started/installation/)
- [Docling — Export tables](https://docling-project.github.io/docling/_generated/examples/export_tables/)
- [Docling — Export figures](https://docling-project.github.io/docling/_generated/examples/export_figures/)
- [Docling — Pipeline options](https://docling-project.github.io/docling/reference/pipeline_options/)
- [Docling — GitHub](https://github.com/docling-project/docling)
- [Docling 2.151.1 — PyPI](https://pypi.org/project/docling/)

**Nota de reproducibilidad:** el notebook fija `docling==2.151.1`. Los modelos pueden descargarse durante la primera conversión. Los resultados pueden variar según la versión, el hardware, la calidad del PDF y la configuración del pipeline.

## Conclusión

El valor de esta información está en enseñar una etapa de ingeniería que muchas aplicaciones de IA necesitan y que suele quedar escondida: **la preparación de los datos antes de que intervenga el modelo.** 🚀